# Agent Evaluation with BFCL-Style Function Calling Metrics

Este notebook presenta un agente de soporte IT impulsado por LLM y lo evalua con una metrica local inspirada en Berkeley Function Calling Leaderboard (BFCL).

El objetivo es medir si el LLM, operando como agente orquestador, rutea a los especialistas correctos y usa las herramientas correctas, con los parametros correctos y en la cantidad correcta.

En este notebook se muestra:
- el dominio y las 13 herramientas disponibles (con enums, arrays y parametros free-text)
- ejemplos de trazas del agente: ruteo, razonamiento resumido, llamadas y respuesta final
- el benchmark local con casos `simple`, `parallel`, `complex`, `relevance` e `irrelevance`
- la evaluacion BFCL-style (matching estricto + tolerante a alias) y metricas de routing

## Agent overview

**Dominio**: soporte IT interno para operaciones frecuentes de mesa de ayuda.

**Capacidades principales** (13 herramientas):
- credenciales: reset de password y desbloqueo de cuenta
- operacion: estado de ticket, estado de servicio, **actualizacion de tickets** y **diagnosticos**
- gestion: creacion de incidentes y escalacion
- conocimiento: busqueda de runbooks o articulos KB
- operacion programada: scheduling de mantenimiento
- **provisioning**: alta de usuarios, **otorgar accesos/roles** (parametro array) y **pedidos de hardware**

**Diseño del agente**: el agente es LLM-driven con OpenAI tool calling y orquestacion LangGraph en tres pasos:

1. **Routing con LLM**: un clasificador LLM decide que especialistas (`identity` / `operations` / `knowledge` / `provisioning`) son relevantes. Cada especialista *posee* un subconjunto de herramientas.
2. **Orquestacion con tools acotadas**: solo se exponen al LLM las herramientas de los especialistas elegidos. Esto hace que el routing sea **consecuente** (un ruteo equivocado deja al agente sin las herramientas que necesita) y **medible**.
3. **Respuesta final**: resumen operativo de las herramientas seleccionadas.

Las herramientas declaran **enums** (incluyendo enums sobre elementos de arrays como `roles`, `tags`, `checks`, `groups`), parametros **free-text** (`description`, `justification`, `comment`) y un **vocabulario controlado** que mapea el lenguaje del usuario al canonico. Esto sube la dificultad: el agente debe rutear bien entre 4 especialistas, completar herramientas con muchos parametros y armar arrays con valores validos.

## Runtime config (LLM required)

Para ejecutar este notebook necesitas variables de entorno configuradas (por ejemplo en `.env` o `.envrc`):
- `OPENAI_API_KEY=...`
- `OPENAI_MODEL=gpt-4.1-mini`
- `OPENAI_TEMPERATURE=0`

In [1]:
import importlib
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Tuple

import pandas as pd
from IPython.display import display

import agent_architecture.config as config_module
import agent_architecture.factory as factory_module
import agent_architecture.tools as tools_module

importlib.reload(config_module)
importlib.reload(tools_module)
importlib.reload(factory_module)

ROOT = Path('.')
CASES_PATH = ROOT / 'agent_architecture' / 'bfcl_like_cases.json'

with open(CASES_PATH, 'r', encoding='utf-8') as f:
    cases = json.load(f)

agent = factory_module.build_agent()

schemas = tools_module.get_tool_schemas()
schema_by_name = {schema.name: schema for schema in schemas}

backend_name = type(agent).__name__
print(f'Loaded {len(cases)} benchmark cases.')
print(f'Loaded {len(schema_by_name)} tool schemas.')
print(f'Agent backend in use: {backend_name}')

Loaded 31 benchmark cases.
Loaded 13 tool schemas.
Agent backend in use: OpenAILangGraphITSupportAgent


In [2]:
# Build readable tables that explain the agent context before evaluating it.
tool_rows = []
for schema in schemas:
    tool_rows.append({
        'tool_name': schema.name,
        'specialist': tools_module.tool_owner(schema.name),
        'description': schema.description,
        'required_params': ', '.join(schema.required),
        'enum_params': ', '.join(schema.enum.keys()),
        'freeform_params': ', '.join(schema.freeform),
    })

tool_catalog_df = pd.DataFrame(tool_rows).sort_values(['specialist', 'tool_name']).reset_index(drop=True)
benchmark_overview_df = pd.DataFrame(cases)[['id', 'category', 'user_query']]

print('Tool catalog used by the agent (tools are exposed per query based on routing)')
display(tool_catalog_df)

print('Benchmark overview')
display(benchmark_overview_df.head(10))
print(benchmark_overview_df['category'].value_counts())

Tool catalog used by the agent (tools are exposed per query based on routing)


,tool_name,specialist,description,required_params,enum_params,freeform_params
0,it_support.reset_password,identity_specialist,Resets a user password and optionally notifies...,username,channel,
1,it_support.unlock_account,identity_specialist,Unlocks a user account in the identity or acce...,username,system,
2,it_support.search_kb_article,knowledge_specialist,Searches the internal knowledge base for runbo...,topic,product,
3,it_support.check_ticket_status,operations_specialist,Returns the current status of an incident ticket.,ticket_id,,
4,it_support.create_incident,operations_specialist,Creates a new incident ticket with service and...,"service, severity, description","service, severity, environment",description
5,it_support.escalate_ticket,operations_specialist,Escalates a ticket to a specialized assignment...,"ticket_id, assignment_group, priority","assignment_group, priority",
6,it_support.get_service_status,operations_specialist,Checks current health for an IT service.,service,service,
7,it_support.run_diagnostic,operations_specialist,Runs health diagnostics on a service for one o...,"service, checks","service, checks",
8,it_support.schedule_maintenance,operations_specialist,Schedules a maintenance window for a service.,"service, start_time, duration_minutes, reason",service,reason
9,it_support.update_ticket,operations_specialist,"Updates an existing ticket: status, assignee, ...",ticket_id,"status, tags",comment


Benchmark overview


,id,category,user_query
0,simple_001,simple,Resetea la contraseña del usuario maria.sosa p...
1,simple_002,simple,Necesito el estado del ticket inc-1203
2,simple_003,simple,Quiero abrir un incidente critica para correo ...
3,simple_004,simple,Dime el estado del servicio vpn en us-east
4,simple_005,simple,Busca una guia KB para configurar MFA en vpn
5,simple_006,simple,Programa mantenimiento de sap el 2026-06-10 22...
6,parallel_001,parallel,Revisa ticket inc-7788 y ademas estado del ser...
7,parallel_002,parallel,Resetea la contraseña del usuario ana.lopez y ...
8,parallel_003,parallel,Consulta el ticket inc-9001 y escalalo al equi...
9,parallel_004,parallel,Abre un incidente alta para jira en qa porque ...


category
parallel       10
simple          6
relevance       5
irrelevance     5
complex         5
Name: count, dtype: int64


## What the agent trace shows

Cada ejecucion del agente devuelve una traza breve con estas piezas:
- **entities**: valores extraidos del pedido del usuario
- **reasoning_steps**: incluye la decision del **router LLM** (que especialistas se eligieron) y el resumen de orquestacion
- **tool_calls**: llamadas estructuradas en formato compatible con la evaluacion, ya con valores canonicos
- **final_response**: resumen textual de lo que el agente planea hacer

Esto sirve para demostrar claramente que se esta evaluando: el ruteo de especialistas (que ahora acota las herramientas disponibles), la toma de decisiones y el uso de herramientas, no solo el texto final.

In [3]:
# Run representative examples so the agent behavior is visible before scoring.
sample_queries = [
    'Resetea la contraseña del usuario maria.sosa por sms, es urgente',
    'Consulta el ticket inc-9001 y escalalo al equipo de correo con prioridad alta',
    'Busca una guia KB de sap y programa mantenimiento de sap el 2026-06-15 23:30 por 120 min',
]

trace_rows = []
for query in sample_queries:
    trace = agent.run(query)
    trace_rows.append({
        'query': trace['query'],
        'entities': json.dumps(trace['entities'], ensure_ascii=False),
        'reasoning_steps': ' | '.join(trace['reasoning_steps']),
        'tool_calls': json.dumps(trace['tool_calls'], ensure_ascii=False),
        'final_response': trace['final_response'],
    })

trace_df = pd.DataFrame(trace_rows)
display(trace_df)

focused_case = cases[8]
focused_trace = agent.run(focused_case['user_query'])

print('Focused trace query:')
print(focused_case['user_query'])
print()
print('Reasoning steps:')
for step_number, step in enumerate(focused_trace['reasoning_steps'], start=1):
    print(f'{step_number}. {step}')
print()
print('Tool calls:')
print(json.dumps(focused_trace['tool_calls'], indent=2, ensure_ascii=False))
print()
print('Final response:')
print(focused_trace['final_response'])

,query,entities,reasoning_steps,tool_calls,final_response
0,Resetea la contraseña del usuario maria.sosa p...,"{""username"": ""maria.sosa"", ""channel"": ""sms"", ""...",LLM router selected: identity_specialist | LLM...,"[{""it_support.reset_password"": {""username"": ""m...",Plan operativo generado con las siguientes her...
1,Consulta el ticket inc-9001 y escalalo al equi...,"{""ticket_id"": ""INC-9001"", ""assignment_group"": ...",LLM router selected: operations_specialist | L...,"[{""it_support.check_ticket_status"": {""ticket_i...",Plan operativo generado con las siguientes her...
2,Busca una guia KB de sap y programa mantenimie...,"{""topic"": ""sap connectivity"", ""product"": ""sap""...","LLM router selected: operations_specialist, kn...","[{""it_support.search_kb_article"": {""topic"": ""s...",Plan operativo generado con las siguientes her...


Focused trace query:
Consulta el ticket inc-9001 y escalalo al equipo de correo con prioridad alta

Reasoning steps:
1. LLM router selected: operations_specialist
2. LLM orchestration summary: Selected tool calls: it_support.check_ticket_status, it_support.escalate_ticket

Tool calls:
[
  {
    "it_support.check_ticket_status": {
      "ticket_id": "INC-9001"
    }
  },
  {
    "it_support.escalate_ticket": {
      "ticket_id": "INC-9001",
      "assignment_group": "messaging_ops",
      "priority": "p2"
    }
  }
]

Final response:
Plan operativo generado con las siguientes herramientas: it_support.check_ticket_status, it_support.escalate_ticket


## BFCL-style evaluation criteria used here

La implementacion local replica la idea central de BFCL para function calling AST-style. Para considerar un caso como correcto, la salida del agente debe cumplir todo esto:

1. **Nombre de funcion correcto**
2. **Parametros requeridos presentes**
3. **Sin parametros inesperados**
4. **Valores dentro del enum** cuando el parametro declara valores permitidos
5. **Tipo y valor correctos**, con **matching tolerante a alias** (mapea `correo`→`email`, `alta`→`p2`, etc. usando el mismo vocabulario que se le enseña al agente) y normalizacion de fechas
6. **Parametros free-text** (p. ej. `description`, `reason`): solo se exige que esten presentes y no vacios, no un match exacto
7. **Matching all-or-nothing** cuando hay multiples llamadas
8. **Irrelevance** correcta cuando el caso espera cero herramientas; **relevance** cuando el pedido esta fraseado de forma casual pero igual requiere una herramienta

Ademas de la accuracy estricta, calculamos:
- **precision/recall de seleccion de herramientas**, over-calling y under-calling
- **routing accuracy/precision/recall**: comparamos los especialistas elegidos por el router contra los *dueños* de las herramientas esperadas (ground truth derivado de la propiedad de tools). Esto mide directamente si el ruteo dejo de ser decorativo.

In [4]:
# Shared controlled vocabulary, imported from the agent package so the evaluator
# scores against exactly the same canonical values the agent was taught to emit.
from agent_architecture.tools import VALUE_ALIASES, KB_TOPICS, canonicalize_datetime


def normalize_string(value: str) -> str:
    # BFCL-like normalization for strings: case-insensitive, ignoring spaces and selected punctuation.
    normalized = value.lower()
    normalized = re.sub(r'[\s,./\-_*^]+', '', normalized)
    return normalized


def canonical_value(param_name: str, value: Any) -> Any:
    # Map a predicted value onto its canonical form using the per-parameter alias
    # table (e.g. correo -> email, alta -> p2). Non-strings pass through unchanged.
    if not isinstance(value, str):
        return value
    aliases = VALUE_ALIASES.get(param_name, {})
    return aliases.get(value.strip().lower(), value)


def normalized_multiset(items: List[Any]) -> List[Any]:
    # Order-insensitive comparison key for array parameters.
    return sorted(normalize_string(x) if isinstance(x, str) else x for x in items)


def is_type_compatible(expected_type: str, value: Any) -> bool:
    if expected_type == 'string':
        return isinstance(value, str)
    if expected_type == 'boolean':
        return isinstance(value, bool)
    if expected_type == 'integer':
        return isinstance(value, int) and not isinstance(value, bool)
    if expected_type == 'float':
        return isinstance(value, (float, int)) and not isinstance(value, bool)
    if expected_type == 'array':
        return isinstance(value, list)
    if expected_type == 'dict':
        return isinstance(value, dict)
    return True


def value_match(param_name: str, expected_candidates: List[Any], value: Any) -> bool:
    non_empty_candidates = [candidate for candidate in expected_candidates if candidate != '']
    if not non_empty_candidates:
        return True

    mapped = canonical_value(param_name, value)
    for candidate in non_empty_candidates:
        if isinstance(candidate, str) and isinstance(value, str):
            # Alias-tolerant comparison against the canonical expected value.
            if normalize_string(mapped) == normalize_string(candidate):
                return True
            # Datetime values compare by digits only (2026-06-10 22:00 == 2026-06-10T22:00).
            if param_name in ('start_time', 'start_date') and canonicalize_datetime(value) == canonicalize_datetime(candidate):
                return True
            # KB topics: accept a close match against the controlled topic catalog.
            if param_name == 'topic':
                norm_value, norm_candidate = normalize_string(mapped), normalize_string(candidate)
                if norm_value and (norm_value in norm_candidate or norm_candidate in norm_value):
                    return True
        elif isinstance(candidate, list) and isinstance(value, list):
            # Array params: alias-map each element, compare order-insensitively.
            mapped_elements = [canonical_value(param_name, element) for element in value]
            if normalized_multiset(mapped_elements) == normalized_multiset(candidate):
                return True
        elif isinstance(candidate, dict) and isinstance(value, dict):
            if candidate == value:
                return True
        else:
            if candidate == value:
                return True

    return False


def validate_single_call(
    expected_call: Dict[str, Dict[str, List[Any]]],
    predicted_call: Dict[str, Dict[str, Any]],
    schema_by_name: Dict[str, Any],
) -> Tuple[bool, str]:
    expected_function = list(expected_call.keys())[0]
    expected_params = expected_call[expected_function]

    if expected_function not in predicted_call:
        return False, f'Wrong function name. Expected {expected_function}.'

    if expected_function not in schema_by_name:
        return False, f'Unknown function schema for {expected_function}.'

    predicted_params = predicted_call[expected_function]
    schema = schema_by_name[expected_function]

    for required_param in schema.required:
        if required_param not in predicted_params:
            return False, f'Missing required parameter: {required_param}.'

    allowed_params = set(schema.properties.keys())
    for param in predicted_params:
        if param not in allowed_params:
            return False, f'Unexpected parameter: {param}.'

    # Enum enforcement: every predicted value for an enum-constrained parameter
    # (including each element of an array parameter) must resolve, after alias
    # mapping, to one of the allowed canonical values.
    for param, value in predicted_params.items():
        if param not in schema.enum:
            continue
        allowed_norm = {normalize_string(allowed) for allowed in schema.enum[param]}
        elements = value if isinstance(value, list) else [value]
        for element in elements:
            if isinstance(element, str) and normalize_string(canonical_value(param, element)) not in allowed_norm:
                return False, f'Value not in enum for {param}: {element}.'

    for param, candidates in expected_params.items():
        param_is_optional = '' in candidates
        if param not in predicted_params:
            if param_is_optional:
                continue
            return False, f'Missing parameter in prediction: {param}.'

        predicted_value = predicted_params[param]
        expected_type = schema.properties[param]['type']

        if not is_type_compatible(expected_type, predicted_value):
            return False, f'Type mismatch on {param}. Expected {expected_type}.'

        # Free-text parameters (e.g. description, reason, justification, comment) are
        # not value-matched; they only need to be present and non-empty.
        if param in schema.freeform:
            if not (isinstance(predicted_value, str) and predicted_value.strip()):
                return False, f'Freeform parameter {param} must be a non-empty string.'
            continue

        if not value_match(param, candidates, predicted_value):
            return False, f'Value mismatch on {param}. Predicted={predicted_value!r}, expected one of {candidates}.'

    return True, 'ok'


def _best_error_for(expected_call, predicted_calls, matched_indices, fallback_error):
    """When an expected call finds no match, pick the most informative error: the
    one from an unmatched predicted call with the SAME function name (so we report
    the real offending parameter), instead of whatever candidate was tried last."""
    expected_function = list(expected_call.keys())[0]
    for idx, predicted_call in enumerate(predicted_calls):
        if idx in matched_indices:
            continue
        if expected_function in predicted_call:
            _, error = validate_single_call(expected_call, predicted_call, schema_by_name)
            return error
    return fallback_error


def evaluate_case(case: Dict[str, Any], predicted_calls: List[Dict[str, Dict[str, Any]]]) -> Dict[str, Any]:
    case_id = case['id']
    category = case['category']
    expected_calls = case['expected_calls']

    if category == 'irrelevance':
        valid = len(predicted_calls) == 0
        return {
            'id': case_id,
            'category': category,
            'valid': valid,
            'error': '' if valid else 'Expected no function call but got at least one.'
        }

    if len(predicted_calls) != len(expected_calls):
        return {
            'id': case_id,
            'category': category,
            'valid': False,
            'error': f'Wrong number of calls. Expected {len(expected_calls)}, got {len(predicted_calls)}.'
        }

    matched_indices = set()
    for expected_call in expected_calls:
        found_match = False
        last_error = 'No candidate prediction available.'

        for idx, predicted_call in enumerate(predicted_calls):
            if idx in matched_indices:
                continue
            valid, error = validate_single_call(expected_call, predicted_call, schema_by_name)
            if valid:
                matched_indices.add(idx)
                found_match = True
                break
            last_error = error

        if not found_match:
            return {
                'id': case_id,
                'category': category,
                'valid': False,
                'error': _best_error_for(expected_call, predicted_calls, matched_indices, last_error),
            }

    return {'id': case_id, 'category': category, 'valid': True, 'error': ''}


def extract_function_names(calls: List[Dict[str, Dict[str, Any]]]) -> List[str]:
    return [list(call.keys())[0] for call in calls]


def compute_selection_metrics(expected_calls, predicted_calls):
    expected_names = extract_function_names(expected_calls)
    predicted_names = extract_function_names(predicted_calls)

    expected_set = set(expected_names)
    predicted_set = set(predicted_names)
    intersection = expected_set & predicted_set

    if not predicted_names and not expected_names:
        precision = 1.0
        recall = 1.0
    else:
        precision = len(intersection) / len(predicted_set) if predicted_set else 0.0
        recall = len(intersection) / len(expected_set) if expected_set else 0.0

    overcalling = int(len(predicted_set - expected_set) > 0 or len(predicted_calls) > len(expected_calls))
    undercalling = int(len(expected_set - predicted_set) > 0 or len(predicted_calls) < len(expected_calls))

    return {
        'expected_function_names': expected_names,
        'predicted_function_names': predicted_names,
        'tool_precision': precision,
        'tool_recall': recall,
        'overcalling': overcalling,
        'undercalling': undercalling,
    }


def expected_specialists_for(function_names: List[str]) -> set:
    # Derive which specialists *should* have been routed, from the owners of the
    # tools each case expects. This is the ground truth for the routing metric.
    specialists = set()
    for function_name in function_names:
        owner = tools_module.tool_owner(function_name)
        if owner:
            specialists.add(owner)
    return specialists


def compute_routing_metrics(expected_function_names: List[str], selected_specialists: List[str]):
    expected = expected_specialists_for(expected_function_names)
    selected = set(selected_specialists)
    intersection = expected & selected

    precision = len(intersection) / len(selected) if selected else 1.0
    recall = len(intersection) / len(expected) if expected else 1.0

    return {
        'expected_specialists': sorted(expected),
        'selected_specialists': sorted(selected),
        'routing_exact': int(expected == selected),
        'routing_precision': precision,
        'routing_recall': recall,
    }

In [5]:
# Evaluate every benchmark case: strict BFCL-style outcome, tool-selection metrics,
# and routing metrics (did the router pick the specialists that own the needed tools?).
rows = []

for case in cases:
    trace = agent.run(case['user_query'])
    predicted_calls = trace['tool_calls']
    strict_result = evaluate_case(case, predicted_calls)
    selection_metrics = compute_selection_metrics(case['expected_calls'], predicted_calls)
    routing_metrics = compute_routing_metrics(
        selection_metrics['expected_function_names'],
        trace['selected_specialists'],
    )

    rows.append({
        'id': case['id'],
        'category': case['category'],
        'query': case['user_query'],
        'expected_calls': json.dumps(case['expected_calls'], ensure_ascii=False),
        'predicted_calls': json.dumps(predicted_calls, ensure_ascii=False),
        'entities': json.dumps(trace['entities'], ensure_ascii=False),
        'reasoning_steps': ' | '.join(trace['reasoning_steps']),
        'final_response': trace['final_response'],
        'valid': strict_result['valid'],
        'error': strict_result['error'],
        **selection_metrics,
        **routing_metrics,
    })

df_eval = pd.DataFrame(rows)
display(df_eval[['id', 'category', 'valid', 'expected_function_names', 'predicted_function_names', 'routing_exact', 'error']])

,id,category,valid,expected_function_names,predicted_function_names,routing_exact,error
0,simple_001,simple,True,[it_support.reset_password],[it_support.reset_password],1,
1,simple_002,simple,True,[it_support.check_ticket_status],[it_support.check_ticket_status],1,
2,simple_003,simple,True,[it_support.create_incident],[it_support.create_incident],1,
3,simple_004,simple,True,[it_support.get_service_status],[it_support.get_service_status],1,
4,simple_005,simple,True,[it_support.search_kb_article],[it_support.search_kb_article],1,
5,simple_006,simple,True,[it_support.schedule_maintenance],[it_support.schedule_maintenance],1,
6,parallel_001,parallel,True,"[it_support.check_ticket_status, it_support.ge...","[it_support.check_ticket_status, it_support.ge...",1,
7,parallel_002,parallel,False,"[it_support.reset_password, it_support.unlock_...","[it_support.reset_password, it_support.unlock_...",1,"Wrong number of calls. Expected 3, got 2."
8,parallel_003,parallel,True,"[it_support.check_ticket_status, it_support.es...","[it_support.check_ticket_status, it_support.es...",1,
9,parallel_004,parallel,True,"[it_support.create_incident, it_support.get_se...","[it_support.create_incident, it_support.get_se...",1,


In [6]:
import json as _json

# Evitar que pandas trunque las columnas con las listas de herramientas.
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


def _fmt_predicted_calls(calls):
    """Herramientas REALMENTE llamadas, una por linea, con sus parametros."""
    if not calls:
        return ['(sin tool calls)']
    lines = []
    for i, call in enumerate(calls, 1):
        fn = list(call.keys())[0]
        params = call[fn]
        args = ', '.join(f'{k}={v!r}' for k, v in params.items())
        lines.append(f'{i}. {fn}({args})')
    return lines


def _fmt_expected_calls(calls):
    """Herramientas ESPERADAS. '?' marca un parametro opcional (el modelo puede omitirlo);
    si hay varios valores aceptables se muestran como lista."""
    if not calls:
        return ['(cero herramientas esperadas)']
    lines = []
    for i, call in enumerate(calls, 1):
        fn = list(call.keys())[0]
        params = call[fn]
        args = []
        for key, candidates in params.items():
            options = [c for c in candidates if c != '']
            optional = '' in candidates
            shown = options[0] if len(options) == 1 else options
            args.append(f'{key}{"?" if optional else ""}={shown!r}')
        lines.append(f'{i}. {fn}(' + ', '.join(args) + ')')
    return lines


# 1) Vista compacta de TODOS los casos: que tools se esperaban vs cuales se llamaron.
overview = df_eval[['id', 'category', 'valid', 'expected_function_names', 'predicted_function_names', 'error']].copy()
overview['valid'] = overview['valid'].map({True: 'OK', False: 'FALLO'})
overview = overview.rename(columns={
    'expected_function_names': 'tools_esperadas',
    'predicted_function_names': 'tools_llamadas',
    'error': 'motivo_fallo',
})
print('Resumen por caso (tools esperadas vs tools llamadas)')
display(overview)

# 2) Detalle legible SOLO de los fallos: aca se ve exactamente donde se rompe cada caso.
failures = df_eval.loc[~df_eval['valid']]
print('\n' + '=' * 92)
print(f'DETALLE DE FALLOS: {len(failures)} de {len(df_eval)} casos   ( ? = parametro opcional )')
print('=' * 92)
for _, row in failures.iterrows():
    print(f"\nFALLO  {row['id']}  [{row['category']}]")
    print(f"  Query: {row['query']}")
    print(f"  Routing  -> esperado: {row['expected_specialists']}  |  elegido: {row['selected_specialists']}")
    print("  Herramientas ESPERADAS:")
    for line in _fmt_expected_calls(_json.loads(row['expected_calls'])):
        print(f"      {line}")
    print("  Herramientas LLAMADAS por el agente:")
    for line in _fmt_predicted_calls(_json.loads(row['predicted_calls'])):
        print(f"      {line}")
    print(f"  >> Motivo del fallo: {row['motivo_fallo'] if 'motivo_fallo' in row else row['error']}")
if failures.empty:
    print('\n(no hubo fallos en esta corrida; reejecuta para ver la variabilidad del modelo)')

Resumen por caso (tools esperadas vs tools llamadas)


,id,category,valid,tools_esperadas,tools_llamadas,motivo_fallo
0,simple_001,simple,OK,[it_support.reset_password],[it_support.reset_password],
1,simple_002,simple,OK,[it_support.check_ticket_status],[it_support.check_ticket_status],
2,simple_003,simple,OK,[it_support.create_incident],[it_support.create_incident],
3,simple_004,simple,OK,[it_support.get_service_status],[it_support.get_service_status],
4,simple_005,simple,OK,[it_support.search_kb_article],[it_support.search_kb_article],
5,simple_006,simple,OK,[it_support.schedule_maintenance],[it_support.schedule_maintenance],
6,parallel_001,parallel,OK,"[it_support.check_ticket_status, it_support.get_service_status]","[it_support.check_ticket_status, it_support.get_service_status]",
7,parallel_002,parallel,FALLO,"[it_support.reset_password, it_support.unlock_account, it_support.check_ticket_status]","[it_support.reset_password, it_support.unlock_account]","Wrong number of calls. Expected 3, got 2."
8,parallel_003,parallel,OK,"[it_support.check_ticket_status, it_support.escalate_ticket]","[it_support.check_ticket_status, it_support.escalate_ticket]",
9,parallel_004,parallel,OK,"[it_support.create_incident, it_support.get_service_status]","[it_support.create_incident, it_support.get_service_status]",



DETALLE DE FALLOS: 2 de 31 casos   ( ? = parametro opcional )

FALLO  parallel_002  [parallel]
  Query: Resetea la contraseña del usuario ana.lopez y desbloquea la cuenta, despues revisa ticket inc-5512
  Routing  -> esperado: ['identity_specialist', 'operations_specialist']  |  elegido: ['identity_specialist', 'operations_specialist']
  Herramientas ESPERADAS:
      1. it_support.reset_password(username='ana.lopez', channel?='email', urgent?=False)
      2. it_support.unlock_account(username='ana.lopez', system?='identity')
      3. it_support.check_ticket_status(ticket_id='INC-5512')
  Herramientas LLAMADAS por el agente:
      1. it_support.reset_password(username='ana.lopez')
      2. it_support.unlock_account(username='ana.lopez')
  >> Motivo del fallo: Wrong number of calls. Expected 3, got 2.

FALLO  parallel_010  [parallel]
  Query: Pedi 2 monitores para CC-100 con envio a us-west y tambien 1 dock para CC-100 a us-west
  Routing  -> esperado: ['provisioning_specialist']  |  el

## Glosario de metricas (que significa cada columna)

Las tablas de abajo resumen el desempeño por categoria y en total. Cada metrica mide algo distinto:

| Metrica | Que mide | Como leerla |
|---|---|---|
| **bfcl_style_accuracy** | % de casos **perfectos**: funcion correcta, parametros requeridos, valores dentro del enum, sin parametros de mas y la cantidad de llamadas exacta. Es **all-or-nothing**: un solo error invalida el caso. | Es la metrica principal. 1.0 = todo perfecto; 0.8 = 1 de cada 5 casos tuvo algun error. |
| **tool_precision** | De las herramientas que el agente **llamo**, que fraccion eran realmente necesarias. | Baja cuando el agente **llama de mas** (herramientas que no correspondian). |
| **tool_recall** | De las herramientas **necesarias**, que fraccion efectivamente llamo. | Baja cuando el agente **deja afuera** herramientas que hacian falta. |
| **overcalling** | Fraccion de casos donde llamo **mas** herramientas de las esperadas. | Ideal cerca de 0. |
| **undercalling** | Fraccion de casos donde llamo **menos** herramientas de las esperadas. | Ideal cerca de 0. |
| **routing_accuracy** | % de casos donde el router eligio **exactamente** el conjunto correcto de especialistas. | Mide si la capa de ruteo (que acota las tools disponibles) acierta. |
| **routing_precision** | De los especialistas elegidos, cuantos correspondian. | Baja si el router activa especialistas de mas. |
| **routing_recall** | De los especialistas necesarios, cuantos eligio. | Baja si el router se olvida de un especialista (y deja al agente sin esas tools). |

> Nota: `bfcl_style_accuracy` puede ser baja aunque `tool_precision`/`tool_recall` sean altas. Eso significa que el agente eligio **las herramientas correctas** pero se equivoco en **algun parametro o valor** (ej. un enum, un dato faltante, una fecha). El **detalle de fallos** de la celda anterior muestra exactamente cual fue el error.

In [7]:
summary_by_category = (
    df_eval.groupby('category')[[
        'valid', 'tool_precision', 'tool_recall', 'overcalling', 'undercalling',
        'routing_exact', 'routing_precision', 'routing_recall',
    ]]
    .mean()
    .rename(columns={'valid': 'bfcl_style_accuracy', 'routing_exact': 'routing_accuracy'})
    .reset_index()
)

overall_summary = pd.DataFrame([{
    'overall_bfcl_style_accuracy': df_eval['valid'].mean(),
    'overall_tool_precision': df_eval['tool_precision'].mean(),
    'overall_tool_recall': df_eval['tool_recall'].mean(),
    'overall_overcalling_rate': df_eval['overcalling'].mean(),
    'overall_undercalling_rate': df_eval['undercalling'].mean(),
    'overall_routing_accuracy': df_eval['routing_exact'].mean(),
    'overall_routing_precision': df_eval['routing_precision'].mean(),
    'overall_routing_recall': df_eval['routing_recall'].mean(),
}])

print('Summary by category')
display(summary_by_category)

print('Overall summary')
display(overall_summary)

# Routing detail: where did the router disagree with the tool-ownership ground truth?
routing_detail = df_eval[['id', 'category', 'expected_specialists', 'selected_specialists', 'routing_exact']]
print('Routing decisions (expected specialists come from tool ownership)')
display(routing_detail)

Summary by category


,category,bfcl_style_accuracy,tool_precision,tool_recall,overcalling,undercalling,routing_accuracy,routing_precision,routing_recall
0,complex,1.0,1.0,1.000000,0.0,0.0,1.0,1.0,1.0
1,irrelevance,1.0,1.0,1.000000,0.0,0.0,1.0,1.0,1.0
2,parallel,0.8,1.0,0.966667,0.0,0.1,1.0,1.0,1.0
3,relevance,1.0,1.0,1.000000,0.0,0.0,1.0,1.0,1.0
4,simple,1.0,1.0,1.000000,0.0,0.0,1.0,1.0,1.0


Overall summary


,overall_bfcl_style_accuracy,overall_tool_precision,overall_tool_recall,overall_overcalling_rate,overall_undercalling_rate,overall_routing_accuracy,overall_routing_precision,overall_routing_recall
0,0.935484,1.0,0.989247,0.0,0.032258,1.0,1.0,1.0


Routing decisions (expected specialists come from tool ownership)


,id,category,expected_specialists,selected_specialists,routing_exact
0,simple_001,simple,[identity_specialist],[identity_specialist],1
1,simple_002,simple,[operations_specialist],[operations_specialist],1
2,simple_003,simple,[operations_specialist],[operations_specialist],1
3,simple_004,simple,[operations_specialist],[operations_specialist],1
4,simple_005,simple,[knowledge_specialist],[knowledge_specialist],1
5,simple_006,simple,[operations_specialist],[operations_specialist],1
6,parallel_001,parallel,[operations_specialist],[operations_specialist],1
7,parallel_002,parallel,"[identity_specialist, operations_specialist]","[identity_specialist, operations_specialist]",1
8,parallel_003,parallel,[operations_specialist],[operations_specialist],1
9,parallel_004,parallel,[operations_specialist],[operations_specialist],1


## Como interpretar los resultados

Con 13 herramientas, 4 especialistas y casos `complex`/`parallel` mas dificiles, es **esperable y deseable** ver scores por debajo de 1.0: reflejan errores reales del modelo y hacen util a la metrica.

**Como atribuir un error** (mirando el detalle de fallos + el glosario):
- `routing_accuracy` < 1.0 → el **router** eligio mal los especialistas; al agente le pueden faltar tools (se traduce en `undercalling`).
- `tool_recall` < 1.0 o `undercalling` alto → el agente **dejo afuera** una herramienta necesaria.
- `tool_precision` < 1.0 o `overcalling` alto → el agente **llamo de mas** (tipico en los casos `irrelevance` con distractores).
- `bfcl_style_accuracy` baja con precision/recall = 1.0 → eligio **bien las tools** pero erro un **parametro/valor** (enum invalido, dato faltante, formato de fecha, etc.). El bloque "DETALLE DE FALLOS" muestra el parametro exacto.

**Sobre la reproducibilidad:** el agente llama a un LLM en vivo (incluso con `temperature=0` hay algo de variabilidad), y la metrica es estricta all-or-nothing. Por eso los numeros pueden moverse un poco entre corridas; cada fallo individual queda explicado en el detalle para que se pueda inspeccionar si fue un error genuino del modelo o un caso ambiguo del benchmark.

El benchmark se puede seguir escalando (mas casos de ambiguedad, datos faltantes, multi-turno) manteniendo el mismo formato de salida, el mismo evaluador y las mismas metricas.